# Solution 45: Preference alignment with DPO

Continue the **instruction checkpoint** from assignment 44. Make a frozen reference copy and update only the policy model. This assignment uses **DPO**: an offline chosen/rejected preference pair for the same prompt. DPO is not GRPO; it does not sample new answers or train a reward model. GRPO would be a separate on-policy extension requiring a reward signal.

The offline pairs are a smoke test. For a small real source, manually download the [7.29 MB UltraFeedback preference file](https://huggingface.co/datasets/HuggingFaceH4/ultrafeedback_binarized/resolve/main/data/test_prefs-00000-of-00001.parquet?download=true) and follow [the data guide](../datasets/llm/README.md). This is the publisher's test split repurposed for an educational local split, not a public benchmark.

Implement `DPOTrainer` in the file cell. Basic PyTorch log-softmax, AdamW and clipping are allowed. A `PreferenceDataset` supplies `(chosen_ids, chosen_labels, rejected_ids, rejected_labels)` with the **same prompt**, response-only labels and masked padding. Long prompts retain their latest bytes, reserving half the context for each answer.

- `sequence_logps(model, ids, labels)`: return the **sum**, per example, of shifted next-token log-probabilities over unmasked completion tokens; no prompt or padding terms.
- `dpo_loss(pi_chosen, pi_rejected, ref_chosen, ref_rejected, beta)`: mean `softplus(-beta * ((pi_chosen-pi_rejected)-(ref_chosen-ref_rejected)))`. Require positive beta. No gradient through reference log-probabilities.
- `train_step(policy, reference, optimizer, batch, beta=.1, max_norm=1.0)`: freeze/evaluate the reference; compute four sequence log-probabilities, backpropagate one DPO loss, clip and update only the policy. Return pre-update loss as a float. Do not mutate reference weights.

The notebook saves `artifacts/llm/aligned_reference.pt`. The judge uses controlled models to compare losses, gradients, one optimizer update and reference immutability. Do not expect a tiny preference set to create general alignment or one fixed chat reply.


In [ ]:
from pathlib import Path
from dataclasses import asdict
import torch
from torch.utils.data import DataLoader
from torch_judge.capstone import LLMConfig, ByteTokenizer, PreferenceDataset, read_jsonl, deterministic
from mini_llm_reference import MiniLLM
from sft_trainer_reference import SFTTrainer


In [ ]:
%%writefile dpo_trainer_reference.py
import torch
from torch.nn import functional as F

class DPOTrainer:
    @staticmethod
    def sequence_logps(model, ids, labels):
        logits = model(ids)[:, :-1]
        targets = labels[:, 1:]
        mask = targets != -100
        if not mask.any(dim=1).all():
            raise ValueError('Every response needs at least one target token')
        selected = F.log_softmax(logits, dim=-1).gather(-1, targets.clamp_min(0).unsqueeze(-1)).squeeze(-1)
        return (selected * mask).sum(dim=-1)

    @staticmethod
    def dpo_loss(pi_chosen, pi_rejected, ref_chosen, ref_rejected, beta=.1):
        if beta <= 0:
            raise ValueError('beta must be positive')
        margin = (pi_chosen - pi_rejected) - (ref_chosen.detach() - ref_rejected.detach())
        return F.softplus(-beta * margin).mean()

    @staticmethod
    def train_step(policy, reference, optimizer, batch, beta=.1, max_norm=1.0):
        if policy is reference:
            raise ValueError('Policy and reference must be separate models')
        if beta <= 0:
            raise ValueError('beta must be positive')
        chosen_ids, chosen_labels, rejected_ids, rejected_labels = batch
        reference.eval()
        reference.requires_grad_(False)
        policy.train()
        optimizer.zero_grad(set_to_none=True)
        pi_c = DPOTrainer.sequence_logps(policy, chosen_ids, chosen_labels)
        pi_r = DPOTrainer.sequence_logps(policy, rejected_ids, rejected_labels)
        with torch.no_grad():
            ref_c = DPOTrainer.sequence_logps(reference, chosen_ids, chosen_labels)
            ref_r = DPOTrainer.sequence_logps(reference, rejected_ids, rejected_labels)
        loss = DPOTrainer.dpo_loss(pi_c, pi_r, ref_c, ref_r, beta)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm)
        optimizer.step()
        return float(loss.detach())


In [ ]:
from dpo_trainer_reference import DPOTrainer
from torch_judge import check
check('llm_preference')


In [ ]:
DATA = Path('datasets/llm/preference')
is_demo = not (DATA / 'train.jsonl').exists()
train_path = DATA / ('demo_train.jsonl' if is_demo else 'train.jsonl')
valid_path = DATA / ('demo_validation.jsonl' if is_demo else 'validation.jsonl')
checkpoint = Path('artifacts/llm/instruction_reference.pt')
if not checkpoint.exists():
    raise FileNotFoundError('Run assignment 44 first to create artifacts/llm/instruction_reference.pt')
sft = torch.load(checkpoint, map_location='cpu', weights_only=True)
assert sft['tokenizer'] == 'utf8_byte_v1'
config = LLMConfig(**sft['config'])
train_data = PreferenceDataset(read_jsonl(train_path), config.max_seq_len)
valid_data = PreferenceDataset(read_jsonl(valid_path), config.max_seq_len)
train_loader = DataLoader(train_data, batch_size=2, shuffle=False, num_workers=0)
valid_loader = DataLoader(valid_data, batch_size=2, shuffle=False, num_workers=0)
print(f'DPO: {len(train_data)} train pairs, {len(valid_data)} validation pairs')


def evaluate_dpo(policy, reference, loader, beta=.1):
    policy.eval()
    reference.eval()
    total = count = 0
    with torch.no_grad():
        for chosen_ids, chosen_labels, rejected_ids, rejected_labels in loader:
            pi_c = DPOTrainer.sequence_logps(policy, chosen_ids, chosen_labels)
            pi_r = DPOTrainer.sequence_logps(policy, rejected_ids, rejected_labels)
            ref_c = DPOTrainer.sequence_logps(reference, chosen_ids, chosen_labels)
            ref_r = DPOTrainer.sequence_logps(reference, rejected_ids, rejected_labels)
            total += DPOTrainer.dpo_loss(pi_c, pi_r, ref_c, ref_r, beta).item() * chosen_ids.size(0)
            count += chosen_ids.size(0)
    return total / count


def run_dpo(epochs=4, max_steps=None):
    with deterministic(2028):
        policy, reference = MiniLLM(config).cpu(), MiniLLM(config).cpu()
        policy.load_state_dict(sft['model_state'])
        reference.load_state_dict(sft['model_state'])
        reference.requires_grad_(False).eval()
        optimizer = torch.optim.AdamW(policy.parameters(), lr=3e-4)
        history, steps = [], 0
        for epoch in range(epochs):
            for batch in train_loader:
                DPOTrainer.train_step(policy, reference, optimizer, batch, beta=.1)
                steps += 1
                if max_steps is not None and steps >= max_steps:
                    break
            row = (evaluate_dpo(policy, reference, train_loader), evaluate_dpo(policy, reference, valid_loader))
            history.append(row)
            print(f'DPO epoch {epoch+1}: train={row[0]:.4f}, validation={row[1]:.4f}')
            if max_steps is not None and steps >= max_steps:
                break
        return policy, reference, optimizer, history

policy, reference, optimizer, history = run_dpo(epochs=4 if is_demo else 1)


In [ ]:
checkpoint_path = Path('artifacts/llm/aligned_reference.pt')
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({'config': asdict(config), 'model_state': policy.state_dict(),
            'tokenizer': 'utf8_byte_v1', 'stage': 'dpo'}, checkpoint_path)
print('Saved:', checkpoint_path)
print('Reply:', SFTTrainer.reply(policy, ByteTokenizer(), [{'role':'user','content':'Hi'}]))
